# 3.18 — Gaussian Discriminant Analysis

Gaussian Discriminant Analysis (GDA) is a generative classifier: it first models how each class could have generated the features, then uses Bayes' rule to turn those class-conditional stories into posterior probabilities. In this lesson, we build the class priors, Gaussian likelihoods, covariance estimates, posterior scores, and decision boundaries directly in NumPy so every probability and linear/quadratic boundary is inspectable.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Gaussian Discriminant Analysis one idea at a time. Run each cell in order and read the printed intermediate values — every probability, determinant, covariance, and score is shown so the classifier is not a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, linear algebra, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic points and plots.

### 1. Generative classification: model each class first

GDA does not start by drawing a boundary. It starts by asking a generative question: "if the label were class 0 or class 1, how likely would this feature vector be?" We therefore need class counts, class priors, and one cloud of points per class before Bayes' rule can classify anything.

In [ ]:
X_w = np.array([[0.2, 1.0], [0.8, 1.3], [1.0, 0.4], [1.4, 1.1],
                [3.0, 2.7], [3.5, 3.2], [4.0, 2.9], [3.6, 3.8]])  # two feature columns.
y_w = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # class labels for the two Gaussian clouds.
classes_w = np.array([0, 1])  # explicit class order used throughout the walkthrough.
print("X shape:", X_w.shape)  # 8 examples by 2 features.
print("class counts:", [int(np.sum(y_w == k)) for k in classes_w])  # four points per class.

▶ What you'll see: a tiny two-class dataset with equal class counts.

In [ ]:
pi_w = np.array([np.mean(y_w == k) for k in classes_w])  # maximum-likelihood class priors.
print("class priors π:", pi_w)  # P(y=k) before seeing x.
assert np.allclose(pi_w, [0.5, 0.5])  # equal counts imply equal priors.

▶ What you'll see: both priors are 0.5 because the training data has four examples from each class.

In [ ]:
plt.figure(figsize=(4.4, 3.4))  # compact scatter plot for the two class clouds.
for k_w, color_w in zip(classes_w, ["steelblue", "darkorange"]):  # plot each class separately.
    pts_w = X_w[y_w == k_w]  # select one class cloud.
    plt.scatter(pts_w[:, 0], pts_w[:, 1], s=70, color=color_w, label=f"class {k_w}")  # draw points.
plt.title("1: GDA starts from class-conditional clouds"); plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.legend(); plt.show()

▶ What you'll see: two separated point clouds; GDA will fit one Gaussian story to each cloud.

*Why it's done this way:* Bayes' rule needs two ingredients, $p(x\mid y=k)$ and $\pi_k=p(y=k)$. Counting labels gives the prior because maximum likelihood for a categorical label is just relative frequency; fitting a density per class gives the likelihood because GDA explains where features come from after the class is chosen.

### 2. Fit Gaussian means and a shared covariance

For each class, GDA estimates a mean vector $\mu_k$, the center of that class's feature cloud. In the classic two-class GDA model, both classes share one covariance matrix $\Sigma$, so they may have different centers but the same elliptical shape. The shared covariance is pooled from residuals $x_i-\mu_{y_i}$.

In [ ]:
mu_w = np.vstack([X_w[y_w == k].mean(axis=0) for k in classes_w])  # one mean vector per class.
print("class means:\n", np.round(mu_w, 3))  # inspect centers of the two clouds.
assert np.allclose(np.round(mu_w, 3), [[0.85, 0.95], [3.525, 3.15]])  # verified means.

▶ What you'll see: class 0 is centered near `(0.85, 0.95)` and class 1 near `(3.525, 3.15)`.

In [ ]:
residuals_w = np.vstack([X_w[i] - mu_w[y_w[i]] for i in range(len(X_w))])  # center every point by its own class mean.
Sigma_w = residuals_w.T @ residuals_w / len(X_w)  # maximum-likelihood pooled covariance.
print("pooled covariance Σ:\n", np.round(Sigma_w, 4))  # shared spread/tilt estimate.
assert np.allclose(np.round(Sigma_w, 4), [[0.1572, 0.0144], [0.0144, 0.1425]])  # verified pooled covariance.

▶ What you'll see: a 2×2 covariance matrix whose off-diagonal value shows the feature cloud tilts upward.

In [ ]:
eigvals_w, eigvecs_w = np.linalg.eigh(Sigma_w)  # eigenvectors describe ellipse directions.
print("covariance eigenvalues:", np.round(eigvals_w, 4))  # variances along principal axes.
plt.figure(figsize=(4.4, 3.4))
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=70)  # plot points colored by class.
for m_w in mu_w:  # draw a small principal-axis cross at each mean.
    for val_w, vec_w in zip(eigvals_w, eigvecs_w.T):
        d_w = vec_w * np.sqrt(val_w)  # one standard deviation along this shared axis.
        plt.plot([m_w[0]-d_w[0], m_w[0]+d_w[0]], [m_w[1]-d_w[1], m_w[1]+d_w[1]], color="black", lw=2)
plt.title("2: shared covariance shape at each mean"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()

▶ What you'll see: the same little covariance cross is copied to both class centers.

*Why it's done this way:* The mean is the maximum-likelihood center of a Gaussian cloud. The pooled covariance divides by all examples because the model assumes one shared $\Sigma$ generated residuals for every class; pooling reduces variance and forces the eventual log-odds boundary to be linear rather than quadratic.

### 3. Compute a multivariate Gaussian likelihood

A Gaussian likelihood measures how typical a point is under a class cloud. In two dimensions,

$$p(x\mid y=k)=\frac{1}{2\pi\sqrt{|\Sigma|}}\exp\left(-\frac12(x-\mu_k)^\top\Sigma^{-1}(x-\mu_k)\right).$$

The exponent is a squared Mahalanobis distance: it measures distance in covariance-scaled units, not raw Euclidean units.

In [ ]:
def gaussian_pdf_w(x, mean, cov):  # multivariate Gaussian density from the formula.
    d = len(x)  # feature dimension.
    diff = x - mean  # displacement from the class center.
    inv = np.linalg.inv(cov)  # precision matrix rescales distances by covariance.
    det = np.linalg.det(cov)  # volume term for the Gaussian ellipse.
    exponent = -0.5 * diff @ inv @ diff  # negative half Mahalanobis distance squared.
    normalizer = 1.0 / np.sqrt(((2 * np.pi) ** d) * det)  # density normalization constant.
    return float(normalizer * np.exp(exponent))  # likelihood value.

x_test_w = np.array([2.2, 2.0])  # a point between the two class centers.
likes_w = np.array([gaussian_pdf_w(x_test_w, mu_w[k], Sigma_w) for k in classes_w])  # class likelihoods.
print("likelihoods p(x|k):", np.round(likes_w, 6))  # inspect density under each class.

▶ What you'll see: both likelihoods are tiny densities, but the larger one marks the more typical class.

In [ ]:
dists_w = np.array([(x_test_w - mu_w[k]) @ np.linalg.inv(Sigma_w) @ (x_test_w - mu_w[k]) for k in classes_w])  # squared Mahalanobis distances.
print("Mahalanobis distance²:", np.round(dists_w, 3))  # lower distance means higher Gaussian density.
assert np.argmin(dists_w) == np.argmax(likes_w)  # the nearest class in covariance units has larger likelihood.

▶ What you'll see: the class with smaller covariance-scaled distance has the larger likelihood.

In [ ]:
xx_w, yy_w = np.meshgrid(np.linspace(-0.2, 4.6, 80), np.linspace(0.0, 4.4, 80))  # grid for contouring.
grid_w = np.c_[xx_w.ravel(), yy_w.ravel()]  # 2-D grid as rows.
z0_w = np.array([gaussian_pdf_w(p, mu_w[0], Sigma_w) for p in grid_w]).reshape(xx_w.shape)  # class-0 density.
plt.figure(figsize=(4.6, 3.6))
plt.contour(xx_w, yy_w, z0_w, levels=7, cmap="Blues")  # density rings for class 0.
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=55)  # training points.
plt.scatter([x_test_w[0]], [x_test_w[1]], marker="x", s=100, color="black", label="test x")  # test point.
plt.title("3: Gaussian likelihood contours"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.legend(); plt.show()

▶ What you'll see: elliptical density contours centered on class 0; density falls with Mahalanobis distance.

*Why it's done this way:* The determinant term makes the density integrate to one by accounting for ellipse volume, while $\Sigma^{-1}$ turns correlated, differently-scaled features into comparable distance units. GDA classifies by typicality under these normalized ellipses, not by raw coordinate distance.

### 4. Apply Bayes' rule with log scores

Bayes' rule combines the likelihood with the prior:

$$p(y=k\mid x)=\frac{p(x\mid y=k)\pi_k}{\sum_j p(x\mid y=j)\pi_j}.$$

In code, we usually compute log scores $\log\pi_k+\log p(x\mid y=k)$ first because products of small densities can underflow.

In [ ]:
joint_w = likes_w * pi_w  # unnormalized posterior numerator p(x|k)π_k.
post_w = joint_w / joint_w.sum()  # normalize over classes so probabilities sum to 1.
print("joint scores:", np.round(joint_w, 8))  # inspect unnormalized class evidence.
print("posterior p(k|x):", np.round(post_w, 4))  # inspect normalized probabilities.
assert round(float(post_w.sum()), 6) == 1.0  # posterior probabilities must sum to one.

▶ What you'll see: the two joint scores are converted into posterior probabilities that sum to exactly 1.

In [ ]:
def log_gaussian_w(x, mean, cov):  # log density version for numerical stability.
    d = len(x)
    diff = x - mean
    inv = np.linalg.inv(cov)
    sign, logdet = np.linalg.slogdet(cov)  # stable log determinant.
    return float(-0.5 * (d * np.log(2 * np.pi) + logdet + diff @ inv @ diff))  # log Gaussian.

log_scores_w = np.array([np.log(pi_w[k]) + log_gaussian_w(x_test_w, mu_w[k], Sigma_w) for k in classes_w])  # log π + log likelihood.
log_post_w = np.exp(log_scores_w - np.max(log_scores_w))  # subtract max before exponentiating.
log_post_w = log_post_w / log_post_w.sum()  # normalize stable exponentials.
print("log scores:", np.round(log_scores_w, 3))
print("posterior from logs:", np.round(log_post_w, 4))
assert np.allclose(post_w, log_post_w)  # log computation matches direct Bayes here.

▶ What you'll see: log-space and direct-space posterior probabilities agree on this small example.

In [ ]:
plt.figure(figsize=(4.4, 3.2))
plt.bar(["class 0", "class 1"], post_w, color=["steelblue", "darkorange"])  # posterior mass by class.
plt.ylim(0, 1); plt.ylabel("posterior probability"); plt.title("4: Bayes posterior for the test point"); plt.show()

▶ What you'll see: the taller bar is the predicted class after prior × likelihood evidence is normalized.

*Why it's done this way:* Bayes' rule is a normalization step: likelihoods say which class explains the point, priors say how common each class was before seeing the point, and the denominator makes the result a valid probability distribution. Log scores preserve the same ordering while avoiding fragile tiny products.

### 5. Shared covariance makes the boundary linear

When both classes share $\Sigma$, the quadratic $x^\top\Sigma^{-1}x$ terms cancel in the log-odds. What remains is a linear score: a weight vector times $x$ plus an intercept. This is why classic two-class GDA creates a straight decision boundary even though it came from Gaussian densities.

In [ ]:
invS_w = np.linalg.inv(Sigma_w)  # shared precision matrix.
w_linear_w = invS_w @ (mu_w[1] - mu_w[0])  # linear coefficient for class-1 versus class-0 log-odds.
b_linear_w = -0.5 * mu_w[1] @ invS_w @ mu_w[1] + 0.5 * mu_w[0] @ invS_w @ mu_w[0] + np.log(pi_w[1] / pi_w[0])  # intercept.
print("linear weights:", np.round(w_linear_w, 3))
print("intercept:", round(float(b_linear_w), 3))

▶ What you'll see: GDA has become a linear classifier in log-odds form.

In [ ]:
log_odds_w = float(w_linear_w @ x_test_w + b_linear_w)  # class-1 log posterior odds.
prob1_w = 1 / (1 + np.exp(-log_odds_w))  # logistic transform of log-odds.
print("log-odds class1/class0:", round(log_odds_w, 3))
print("P(class 1 | x) from log-odds:", round(prob1_w, 4))
assert abs(prob1_w - post_w[1]) < 1e-10  # binary posterior matches Bayes calculation.

▶ What you'll see: the logistic transform of the linear GDA score equals the Bayes posterior for class 1.

In [ ]:
xx_line_w = np.linspace(-0.2, 4.6, 100)  # x-axis values for the boundary.
yy_line_w = -(w_linear_w[0] * xx_line_w + b_linear_w) / w_linear_w[1]  # solve w1*x1+w2*x2+b=0.
plt.figure(figsize=(4.8, 3.6))
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=70)  # training clouds.
plt.plot(xx_line_w, yy_line_w, color="black", lw=2, label="GDA boundary")  # linear boundary.
plt.xlim(-0.2, 4.6); plt.ylim(0.0, 4.4); plt.xlabel("feature 1"); plt.ylabel("feature 2")
plt.title("5: shared covariance → linear boundary"); plt.legend(); plt.show()

▶ What you'll see: a straight line separating the two Gaussian clouds.

*Why it's done this way:* Equal covariance says both classes have the same shape, so the only difference is where each Gaussian is centered and how common the class is. Algebraically, that shared shape cancels the quadratic term, leaving a linear log-odds boundary like logistic regression but with parameters estimated generatively.

### 6. Separate covariance gives QDA and a curved boundary

If every class gets its own covariance, the quadratic terms no longer cancel. That model is Quadratic Discriminant Analysis (QDA), a more flexible cousin of GDA. It can represent curved boundaries, but it estimates more covariance parameters and can overfit small classes.

In [ ]:
covs_q_w = []  # separate covariance per class.
for k_w in classes_w:
    centered_w = X_w[y_w == k_w] - mu_w[k_w]  # class-specific residuals.
    covs_q_w.append(centered_w.T @ centered_w / np.sum(y_w == k_w))  # maximum-likelihood class covariance.
covs_q_w = np.array(covs_q_w)  # shape: classes by features by features.
print("class-specific covariances:\n", np.round(covs_q_w, 4))

▶ What you'll see: each class now has its own 2×2 spread matrix instead of sharing the pooled one.

In [ ]:
def qda_log_scores_w(points):  # compute class-specific covariance log scores for many points.
    out = []
    for k in classes_w:
        inv = np.linalg.inv(covs_q_w[k])
        sign, logdet = np.linalg.slogdet(covs_q_w[k])
        diff = points - mu_w[k]
        quad = np.sum((diff @ inv) * diff, axis=1)
        out.append(np.log(pi_w[k]) - 0.5 * (2 * np.log(2 * np.pi) + logdet + quad))
    return np.vstack(out).T

qda_scores_w = qda_log_scores_w(grid_w)  # log scores for every grid point.
qda_pred_w = np.argmax(qda_scores_w, axis=1).reshape(xx_w.shape)  # class with larger quadratic score.
print("QDA prediction at test x:", int(np.argmax(qda_log_scores_w(x_test_w[None, :]))))

▶ What you'll see: QDA still predicts by the larger log score, but the score contains class-specific quadratic distance.

In [ ]:
plt.figure(figsize=(4.8, 3.6))
plt.contourf(xx_w, yy_w, qda_pred_w, levels=[-0.5, 0.5, 1.5], alpha=0.20, colors=["steelblue", "darkorange"])  # decision regions.
plt.contour(xx_w, yy_w, qda_pred_w, levels=[0.5], colors="black", linewidths=2)  # boundary curve.
plt.scatter(X_w[:, 0], X_w[:, 1], c=y_w, cmap="coolwarm", s=70)  # training points.
plt.title("6: separate covariance → quadratic boundary"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()

▶ What you'll see: the decision region can bend because each class has its own covariance shape.

*Why it's done this way:* Separate covariance increases expressiveness by letting each class have a different ellipse. That flexibility is useful when spreads truly differ, but it costs many more parameters; with few examples, covariance estimates can become noisy or singular, so the shared-covariance GDA assumption is often a deliberate stability tradeoff.

### 7. Stabilize covariance estimates before inverting

Every GDA prediction in multiple dimensions needs $\Sigma^{-1}$. If features are redundant or a class has too few points, a covariance matrix can be nearly singular, making the inverse unstable. A small diagonal ridge $\epsilon I$ keeps variances positive and makes the inverse well-behaved.

In [ ]:
X_bad_w = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0], [4.0, 8.0]])  # second feature is exactly 2× the first.
centered_bad_w = X_bad_w - X_bad_w.mean(axis=0)  # center the redundant features.
S_bad_w = centered_bad_w.T @ centered_bad_w / len(X_bad_w)  # singular covariance.
print("singular covariance:\n", S_bad_w)
print("determinant:", round(float(np.linalg.det(S_bad_w)), 10))
assert round(float(np.linalg.det(S_bad_w)), 10) == 0.0  # exact linear dependence makes det zero.

▶ What you'll see: determinant zero, so the covariance ellipse has collapsed into a line.

In [ ]:
eps_w = 0.1  # small diagonal ridge.
S_reg_w = S_bad_w + eps_w * np.eye(2)  # covariance regularization.
print("regularized covariance:\n", np.round(S_reg_w, 3))
print("regularized determinant:", round(float(np.linalg.det(S_reg_w)), 3))
assert np.linalg.det(S_reg_w) > 0  # now invertible.

▶ What you'll see: adding `0.1` to each diagonal variance makes the determinant positive.

In [ ]:
plt.figure(figsize=(4.4, 3.2))
plt.bar(["det(Σ)", "det(Σ+εI)"], [np.linalg.det(S_bad_w), np.linalg.det(S_reg_w)], color=["crimson", "seagreen"])
plt.title("7: diagonal ridge makes covariance invertible"); plt.ylabel("determinant"); plt.show()

▶ What you'll see: the regularized determinant is positive while the unregularized one is zero.

*Why it's done this way:* The inverse covariance is a precision matrix; if a direction has zero variance, precision would be infinite and the Gaussian formula breaks. Adding $\epsilon I$ says every direction has at least a tiny variance, trading a little bias for a classifier that can be computed reliably.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, class counts, covariance matrices, inverses, and assertions.
import matplotlib.pyplot as plt  # load Matplotlib for scatter plots, contours, heatmaps, and diagnostic bars.
np.random.seed(0)  # make all random examples reproducible across notebook runs.

def gda_fit(X, y, shared=True, ridge=1e-6):  # fit priors, means, and covariance parameters for GDA/QDA.
    classes = np.unique(y)  # store class labels in sorted order.
    priors = np.array([np.mean(y == k) for k in classes])  # estimate π_k by class frequency.
    means = np.vstack([X[y == k].mean(axis=0) for k in classes])  # estimate one mean vector per class.
    if shared:  # use one pooled covariance for classic GDA.
        residuals = np.vstack([X[i] - means[np.where(classes == y[i])[0][0]] for i in range(len(y))])  # residual from each point to its class mean.
        cov = residuals.T @ residuals / len(y) + ridge * np.eye(X.shape[1])  # maximum-likelihood pooled covariance plus ridge.
    else:  # use one covariance per class for QDA-style comparison.
        cov = []  # collect class-specific covariance matrices.
        for idx, k in enumerate(classes):  # loop over classes.
            centered = X[y == k] - means[idx]  # class-specific residuals.
            cov.append(centered.T @ centered / np.sum(y == k) + ridge * np.eye(X.shape[1]))  # class covariance plus ridge.
        cov = np.array(cov)  # convert to one array.
    return classes, priors, means, cov  # return fitted generative parameters.

def log_gaussian(x, mean, cov):  # compute log N(x; mean, cov) for one vector.
    x = np.asarray(x, dtype=float)  # ensure numeric array input.
    diff = x - mean  # displacement from the Gaussian center.
    sign, logdet = np.linalg.slogdet(cov)  # stable log determinant for the normalizer.
    inv = np.linalg.inv(cov)  # precision matrix for Mahalanobis distance.
    d = x.size  # feature dimension.
    return float(-0.5 * (d * np.log(2 * np.pi) + logdet + diff @ inv @ diff))  # log density.

def gda_log_scores(X, priors, means, cov):  # compute log π_k + log p(x|k) for each row and class.
    X = np.atleast_2d(X).astype(float)  # accept one point or many points.
    scores = np.zeros((X.shape[0], len(priors)))  # allocate score table.
    for k in range(len(priors)):  # loop over classes.
        C = cov if cov.ndim == 2 else cov[k]  # use shared or class-specific covariance.
        for i in range(X.shape[0]):  # loop over rows for clarity.
            scores[i, k] = np.log(priors[k]) + log_gaussian(X[i], means[k], C)  # prior plus likelihood.
    return scores  # rows are examples, columns are classes.

def softmax_from_log(scores):  # normalize log scores into probabilities.
    shifted = scores - np.max(scores, axis=1, keepdims=True)  # subtract row max for numerical stability.
    exp_scores = np.exp(shifted)  # exponentiate stable scores.
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)  # normalize each row.

def plot_points(X, y, title):  # draw a simple 2-D class scatter plot.
    plt.figure(figsize=(4, 3))  # compact figure.
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=60, edgecolor="k")  # color by label.
    plt.title(title)  # title the plot.
    plt.xlabel("feature 1")  # label x-axis.
    plt.ylabel("feature 2")  # label y-axis.
    plt.show()  # display the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Count classes and estimate priors

**Goal.** Estimate class priors from label frequencies, because GDA uses $\pi_k=P(y=k)$ before looking at features. We build it in 2 steps.

In [ ]:
y_b1 = np.array([0, 0, 0, 1, 1, 1, 1, 1])  # create labels with an imbalanced class distribution.
classes_b1 = np.unique(y_b1)  # find the labels present in the training set.
counts_b1 = np.array([np.sum(y_b1 == k) for k in classes_b1])  # count examples per class.
print("classes:", classes_b1)  # inspect label order.
print("counts:", counts_b1)  # inspect raw frequency evidence.

▶ What you'll see: class 0 has 3 examples and class 1 has 5 examples.

In [ ]:
priors_b1 = counts_b1 / len(y_b1)  # maximum-likelihood estimate of class probabilities.
print("priors:", priors_b1)  # inspect P(y=0) and P(y=1).
assert np.allclose(priors_b1, [0.375, 0.625])  # verify the arithmetic 3/8 and 5/8.
plt.figure(figsize=(4, 3))  # create a compact prior plot.
plt.bar(["class 0", "class 1"], priors_b1, color=["steelblue", "darkorange"])  # visualize prior mass.
plt.ylim(0, 1); plt.title("Basic 1: class priors"); plt.ylabel("π_k"); plt.show()  # display the plot.

▶ What you'll see: the prior bar for class 1 is taller because class 1 is more common.

👀 Takeaway: GDA priors are label frequencies, and they influence predictions even before feature likelihoods are considered.

### Basic 2 — Compute class means

**Goal.** Find one mean vector per class, because each Gaussian class density is centered at $\mu_k$. We build it in 2 steps.

In [ ]:
X_b2 = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 2.0], [4.0, 5.0], [5.0, 5.0], [6.0, 4.0]])  # two clouds.
y_b2 = np.array([0, 0, 0, 1, 1, 1])  # first three points are class 0, last three class 1.
print("class 0 points:\n", X_b2[y_b2 == 0])  # inspect the first cloud.
print("class 1 points:\n", X_b2[y_b2 == 1])  # inspect the second cloud.

▶ What you'll see: each class has three two-dimensional points.

In [ ]:
means_b2 = np.vstack([X_b2[y_b2 == k].mean(axis=0) for k in np.unique(y_b2)])  # compute class centers.
print("means:\n", means_b2)  # inspect μ0 and μ1.
assert np.allclose(means_b2, [[1.0, 4/3], [5.0, 14/3]])  # verify hand-computed means.
plt.figure(figsize=(4, 3))  # create class-center figure.
plt.scatter(X_b2[:, 0], X_b2[:, 1], c=y_b2, cmap="coolwarm", s=60)  # plot data.
plt.scatter(means_b2[:, 0], means_b2[:, 1], marker="x", s=140, color="black")  # plot means.
plt.title("Basic 2: class means μ_k"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()  # show plot.

▶ What you'll see: black x markers sit at the centers of the two point clouds.

👀 Takeaway: a GDA mean is the feature average inside one class, not the average over the whole dataset.

### Basic 3 — Center points by their own class

**Goal.** Compute residuals $x_i-\mu_{y_i}$, because the shared covariance is built from within-class deviations. We build it in 2 steps.

In [ ]:
X_b3 = np.array([[0.0, 1.0], [1.0, 1.0], [2.0, 2.0], [4.0, 5.0], [5.0, 5.0], [6.0, 4.0]])  # reuse a small dataset.
y_b3 = np.array([0, 0, 0, 1, 1, 1])  # class labels.
means_b3 = np.vstack([X_b3[y_b3 == k].mean(axis=0) for k in np.unique(y_b3)])  # class means.
print("means:\n", np.round(means_b3, 3))  # inspect centers before subtracting.

▶ What you'll see: two class centers, one for each cloud.

In [ ]:
res_b3 = np.vstack([X_b3[i] - means_b3[y_b3[i]] for i in range(len(y_b3))])  # residual from each point to its class mean.
print("first three residuals:\n", np.round(res_b3[:3], 3))  # inspect class-0 deviations.
assert np.allclose(np.round(res_b3.sum(axis=0), 10), [0.0, 0.0])  # residuals sum to zero within all centered groups.
plt.figure(figsize=(4, 3))  # create residual plot.
plt.scatter(res_b3[:, 0], res_b3[:, 1], c=y_b3, cmap="coolwarm", s=60)  # plot centered points.
plt.axhline(0, color="black", lw=0.7); plt.axvline(0, color="black", lw=0.7)  # add origin axes.
plt.title("Basic 3: within-class residuals"); plt.xlabel("residual feature 1"); plt.ylabel("residual feature 2"); plt.show()  # display.

▶ What you'll see: both classes are recentered around zero so covariance measures spread, not location.

👀 Takeaway: covariance should measure spread around each class center, not separation between class centers.

### Basic 4 — Pool residuals into one covariance

**Goal.** Estimate the shared covariance matrix, because classic GDA assumes all classes have the same Gaussian shape. We build it in 2 steps.

In [ ]:
res_b4 = np.array([[-1.0, -1/3], [0.0, -1/3], [1.0, 2/3], [-1.0, 1/3], [0.0, 1/3], [1.0, -2/3]])  # centered residuals.
n_b4 = len(res_b4)  # number of training examples.
scatter_b4 = res_b4.T @ res_b4  # sum of outer products before dividing.
print("scatter matrix:\n", np.round(scatter_b4, 3))  # inspect total residual energy.

▶ What you'll see: the scatter matrix summarizes squared residuals and cross-products.

In [ ]:
Sigma_b4 = scatter_b4 / n_b4  # maximum-likelihood pooled covariance.
print("shared covariance:\n", np.round(Sigma_b4, 3))  # inspect Σ.
assert np.allclose(np.round(Sigma_b4, 3), [[0.667, 0.0], [0.0, 0.222]])  # verified diagonal covariance.
plt.figure(figsize=(4, 3))  # create covariance heatmap.
plt.imshow(Sigma_b4, cmap="viridis")  # visualize covariance entries.
plt.colorbar(label="covariance"); plt.title("Basic 4: pooled covariance Σ"); plt.xticks([0, 1]); plt.yticks([0, 1]); plt.show()  # display.

▶ What you'll see: feature 1 has larger variance than feature 2 and the cross-covariance is zero.

👀 Takeaway: pooling residuals produces one covariance matrix that all class likelihoods will share.

### Basic 5 — Read the covariance determinant

**Goal.** Compute $|\Sigma|$, because the Gaussian normalizer uses determinant as an ellipse-volume measure. We build it in 2 steps.

In [ ]:
Sigma_b5 = np.array([[0.667, 0.0], [0.0, 0.222]])  # simple diagonal covariance from the previous idea.
det_b5 = np.linalg.det(Sigma_b5)  # determinant equals product of diagonal entries for diagonal matrices.
print("determinant:", round(float(det_b5), 3))  # inspect ellipse area scale.

▶ What you'll see: the determinant is about 0.148.

In [ ]:
normalizer_b5 = 1 / np.sqrt(((2 * np.pi) ** 2) * det_b5)  # 2-D Gaussian normalization constant.
print("Gaussian normalizer:", round(float(normalizer_b5), 3))  # inspect density scale at zero Mahalanobis distance.
assert round(float(det_b5), 3) == 0.148  # verify determinant arithmetic.
plt.figure(figsize=(4, 3))  # create determinant component plot.
plt.bar(["var f1", "var f2", "det"], [Sigma_b5[0, 0], Sigma_b5[1, 1], det_b5], color="teal")  # compare variances and determinant.
plt.title("Basic 5: determinant as volume"); plt.ylabel("value"); plt.show()  # display plot.

▶ What you'll see: the determinant is smaller than either variance because it multiplies both axes of spread.

👀 Takeaway: a larger covariance volume lowers peak density; a smaller volume makes the Gaussian more concentrated.

### Basic 6 — Compute one Mahalanobis distance

**Goal.** Measure covariance-scaled distance from a class mean, because Gaussian likelihood decays with Mahalanobis distance squared. We build it in 3 steps.

In [ ]:
x_b6 = np.array([2.0, 2.0])  # point to score.
mu_b6 = np.array([1.0, 4/3])  # class mean.
Sigma_b6 = np.array([[2/3, 0.0], [0.0, 2/9]])  # diagonal covariance.
diff_b6 = x_b6 - mu_b6  # displacement from the mean.
print("diff:", np.round(diff_b6, 3))  # inspect raw displacement.

▶ What you'll see: the point is 1.0 unit away in feature 1 and 0.667 in feature 2.

In [ ]:
inv_b6 = np.linalg.inv(Sigma_b6)  # precision matrix.
maha2_b6 = float(diff_b6 @ inv_b6 @ diff_b6)  # squared Mahalanobis distance.
print("Mahalanobis distance²:", round(maha2_b6, 3))  # inspect covariance-scaled distance.
assert round(maha2_b6, 3) == 3.5  # 1^2/(2/3) + (2/3)^2/(2/9) = 1.5 + 2.0.

In [ ]:
plt.figure(figsize=(4, 3))  # create contribution plot.
plt.bar(["feature 1", "feature 2"], diff_b6 ** 2 / np.diag(Sigma_b6), color="purple")  # per-feature standardized distance.
plt.title("Basic 6: contributions to Mahalanobis distance"); plt.ylabel("squared standardized distance"); plt.show()  # display.

▶ What you'll see: feature 2 contributes more because its variance is smaller.

👀 Takeaway: Mahalanobis distance penalizes deviations more strongly in low-variance directions.

### Basic 7 — Evaluate a Gaussian density

**Goal.** Turn distance and determinant into a likelihood value, because GDA needs $p(x\mid y=k)$. We build it in 3 steps.

In [ ]:
x_b7 = np.array([2.0, 2.0])  # point to score.
mu_b7 = np.array([1.0, 4/3])  # Gaussian mean.
Sigma_b7 = np.array([[2/3, 0.0], [0.0, 2/9]])  # Gaussian covariance.
print("x:", x_b7, "mu:", np.round(mu_b7, 3))  # inspect density inputs.

▶ What you'll see: one point, one mean, and one covariance define the likelihood calculation.

In [ ]:
diff_b7 = x_b7 - mu_b7  # displacement.
maha2_b7 = float(diff_b7 @ np.linalg.inv(Sigma_b7) @ diff_b7)  # distance squared.
det_b7 = float(np.linalg.det(Sigma_b7))  # covariance volume.
pdf_b7 = float(np.exp(-0.5 * maha2_b7) / np.sqrt(((2 * np.pi) ** 2) * det_b7))  # 2-D Gaussian density.
print("pdf:", round(pdf_b7, 4))  # inspect likelihood.
assert round(pdf_b7, 4) == 0.0719  # verified density.

In [ ]:
plt.figure(figsize=(4, 3))  # create likelihood ingredients plot.
plt.bar(["maha²", "det", "pdf"], [maha2_b7, det_b7, pdf_b7], color=["red", "gray", "green"])  # compare terms.
plt.title("Basic 7: Gaussian density ingredients"); plt.show()  # display.

▶ What you'll see: a moderate Mahalanobis distance produces a small but nonzero density.

👀 Takeaway: likelihood is high near the mean and falls exponentially with covariance-scaled distance.

### Basic 8 — Combine likelihoods with priors

**Goal.** Compute unnormalized joint scores $p(x\mid y=k)\pi_k$, because Bayes' rule ranks classes by likelihood times prior. We build it in 2 steps.

In [ ]:
likes_b8 = np.array([0.0719, 0.0120])  # pretend class-conditional densities for one point.
priors_b8 = np.array([0.375, 0.625])  # class priors from an imbalanced label set.
joint_b8 = likes_b8 * priors_b8  # Bayes numerator for each class.
print("joint scores:", np.round(joint_b8, 5))  # inspect p(x|k)π_k.

▶ What you'll see: the stronger likelihood for class 0 must still compete with the larger class-1 prior.

In [ ]:
posterior_b8 = joint_b8 / joint_b8.sum()  # normalize into probabilities.
print("posterior:", np.round(posterior_b8, 4))  # inspect P(y=k|x).
assert round(float(posterior_b8.sum()), 6) == 1.0  # probabilities sum to one.
plt.figure(figsize=(4, 3))  # create posterior plot.
plt.bar(["class 0", "class 1"], posterior_b8, color=["steelblue", "darkorange"])  # visualize posterior.
plt.ylim(0, 1); plt.title("Basic 8: posterior from prior × likelihood"); plt.ylabel("probability"); plt.show()  # display.

▶ What you'll see: posterior probabilities sum to one after normalization.

👀 Takeaway: the prior can shift a decision, but it cannot help a class whose likelihood is too small.

### Basic 9 — Normalize log scores safely

**Goal.** Convert log scores to probabilities without underflow, because real Gaussian densities can be extremely small. We build it in 2 steps.

In [ ]:
log_scores_b9 = np.array([[-12.0, -14.0]])  # two class log joint scores for one point.
shifted_b9 = log_scores_b9 - np.max(log_scores_b9, axis=1, keepdims=True)  # subtract max before exponentiating.
print("shifted log scores:", shifted_b9)  # inspect stable values.

▶ What you'll see: the largest score becomes 0, so exponentials stay in a safe numeric range.

In [ ]:
prob_b9 = np.exp(shifted_b9) / np.exp(shifted_b9).sum(axis=1, keepdims=True)  # stable softmax.
print("probabilities:", np.round(prob_b9, 4))  # inspect normalized posterior.
assert np.allclose(np.round(prob_b9, 4), [[0.8808, 0.1192]])  # verified logistic difference of 2.
plt.figure(figsize=(4, 3))  # create stable-softmax plot.
plt.bar(["class 0", "class 1"], prob_b9.ravel(), color=["steelblue", "darkorange"])  # visualize probabilities.
plt.ylim(0, 1); plt.title("Basic 9: stable normalization"); plt.show()  # display.

▶ What you'll see: a log-score gap of 2 gives about 88% versus 12% posterior mass.

👀 Takeaway: subtracting the maximum changes neither probabilities nor the predicted class, but prevents numerical underflow.

### Basic 10 — Predict by maximum posterior

**Goal.** Pick the class with the largest posterior probability, because GDA classification is Bayes posterior maximization. We build it in 2 steps.

In [ ]:
posterior_b10 = np.array([[0.27, 0.73], [0.61, 0.39], [0.50, 0.50]])  # posterior rows for three examples.
pred_b10 = np.argmax(posterior_b10, axis=1)  # choose the highest-probability class in each row.
print("predicted classes:", pred_b10)  # inspect class decisions.

▶ What you'll see: each row is assigned to its largest posterior probability.

In [ ]:
confidence_b10 = np.max(posterior_b10, axis=1)  # probability assigned to the predicted class.
print("confidence:", confidence_b10)  # inspect how decisive each prediction is.
assert np.array_equal(pred_b10, [1, 0, 0])  # ties go to the first maximum in NumPy's argmax.
plt.figure(figsize=(4, 3))  # create confidence plot.
plt.bar(["x0", "x1", "x2"], confidence_b10, color="seagreen")  # visualize decision confidence.
plt.ylim(0, 1); plt.title("Basic 10: posterior confidence"); plt.ylabel("max posterior"); plt.show()  # display.

▶ What you'll see: the tie has only 0.5 confidence, unlike the first two examples.

👀 Takeaway: GDA predicts with `argmax` over posterior probabilities, while the maximum value gives a useful confidence diagnostic.

## 🟡 Easy

### Easy 1 — Fit a complete two-class GDA model

**Goal.** Use the setup helpers to estimate priors, means, and a pooled covariance, because these are the sufficient statistics of classic GDA. We build it in 3 steps.

In [ ]:
X_e1 = np.array([[0.2, 1.0], [0.8, 1.3], [1.0, 0.4], [1.4, 1.1], [3.0, 2.7], [3.5, 3.2], [4.0, 2.9], [3.6, 3.8]])  # training features.
y_e1 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # training labels.
classes_e1, priors_e1, means_e1, cov_e1 = gda_fit(X_e1, y_e1, shared=True, ridge=0.0)  # fit shared-covariance GDA.
print("priors:", priors_e1)  # inspect class frequencies.

▶ What you'll see: equal priors because both classes have four training examples.

In [ ]:
print("means:\n", np.round(means_e1, 3))  # inspect class centers.
print("covariance:\n", np.round(cov_e1, 4))  # inspect pooled covariance.
assert np.allclose(np.round(means_e1, 3), [[0.85, 0.95], [3.525, 3.15]])  # verify fitted means.
assert np.allclose(np.round(cov_e1, 4), [[0.1572, 0.0144], [0.0144, 0.1425]])  # verify covariance.

In [ ]:
plot_points(X_e1, y_e1, "Easy 1: fitted GDA training clouds")  # visualize the fitted dataset.

▶ What you'll see: two clouds whose centers and pooled spread were just estimated.

👀 Takeaway: fitting GDA is mostly counting labels, averaging class features, and pooling centered residuals.

### Easy 2 — Classify new points with Bayes posteriors

**Goal.** Score new examples with log priors plus log Gaussian likelihoods, because GDA predictions are posterior probabilities. We build it in 3 steps.

In [ ]:
X_e2 = np.array([[0.2, 1.0], [0.8, 1.3], [1.0, 0.4], [1.4, 1.1], [3.0, 2.7], [3.5, 3.2], [4.0, 2.9], [3.6, 3.8]])  # training data.
y_e2 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
_, priors_e2, means_e2, cov_e2 = gda_fit(X_e2, y_e2, shared=True, ridge=1e-6)  # fit stable GDA.
X_new_e2 = np.array([[1.0, 1.0], [3.4, 3.0], [2.2, 2.0]])  # points to classify.
print("new points:\n", X_new_e2)  # inspect prediction inputs.

▶ What you'll see: one point near class 0, one near class 1, and one in between.

In [ ]:
scores_e2 = gda_log_scores(X_new_e2, priors_e2, means_e2, cov_e2)  # compute log posterior numerators.
probs_e2 = softmax_from_log(scores_e2)  # normalize into posterior probabilities.
pred_e2 = np.argmax(probs_e2, axis=1)  # choose maximum posterior class.
print("posterior probabilities:\n", np.round(probs_e2, 4))  # inspect probabilities.
print("predictions:", pred_e2)  # inspect class choices.
assert np.array_equal(pred_e2, [0, 1, 0])  # verified decisions for these points.

In [ ]:
plt.figure(figsize=(4.5, 3.3))  # create prediction scatter plot.
plt.scatter(X_e2[:, 0], X_e2[:, 1], c=y_e2, cmap="coolwarm", s=55, alpha=0.65)  # training points.
plt.scatter(X_new_e2[:, 0], X_new_e2[:, 1], c=pred_e2, cmap="coolwarm", marker="x", s=120, linewidths=3)  # new predictions.
plt.title("Easy 2: GDA predictions for new points"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()  # display.

▶ What you'll see: x markers near each cloud get that cloud's label; the middle point is assigned by posterior evidence.

👀 Takeaway: GDA classification is just posterior scoring after the generative parameters are fitted.

### Easy 3 — Draw the linear GDA decision boundary

**Goal.** Visualize the class-1 posterior across a grid, because shared covariance turns GDA into a linear log-odds classifier. We build it in 3 steps.

In [ ]:
X_e3 = np.array([[0.2, 1.0], [0.8, 1.3], [1.0, 0.4], [1.4, 1.1], [3.0, 2.7], [3.5, 3.2], [4.0, 2.9], [3.6, 3.8]])  # training features.
y_e3 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # training labels.
_, priors_e3, means_e3, cov_e3 = gda_fit(X_e3, y_e3, shared=True, ridge=1e-6)  # fit GDA.
xx_e3, yy_e3 = np.meshgrid(np.linspace(-0.2, 4.6, 90), np.linspace(0.0, 4.4, 90))  # grid for decision regions.
grid_e3 = np.c_[xx_e3.ravel(), yy_e3.ravel()]  # convert grid to row format.
print("grid points:", grid_e3.shape[0])  # inspect grid size.

▶ What you'll see: thousands of grid points will be scored to reveal the boundary.

In [ ]:
prob1_e3 = softmax_from_log(gda_log_scores(grid_e3, priors_e3, means_e3, cov_e3))[:, 1].reshape(xx_e3.shape)  # class-1 posterior on grid.
print("posterior range:", round(float(prob1_e3.min()), 3), "to", round(float(prob1_e3.max()), 3))  # inspect probability span.
assert prob1_e3.min() < 0.01 and prob1_e3.max() > 0.99  # grid covers confident regions for both classes.

In [ ]:
plt.figure(figsize=(4.8, 3.6))  # create boundary figure.
plt.contourf(xx_e3, yy_e3, prob1_e3, levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.35)  # posterior background.
plt.contour(xx_e3, yy_e3, prob1_e3, levels=[0.5], colors="black", linewidths=2)  # decision boundary.
plt.scatter(X_e3[:, 0], X_e3[:, 1], c=y_e3, cmap="coolwarm", s=60, edgecolor="k")  # data points.
plt.colorbar(label="P(class 1 | x)"); plt.title("Easy 3: linear GDA boundary"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()  # display.

▶ What you'll see: the 0.5 contour is a straight line because the two classes share covariance.

👀 Takeaway: shared covariance makes GDA's posterior log-odds linear in the input features.

### Easy 4 — Compare GDA and QDA covariances

**Goal.** Fit shared and class-specific covariance models, because GDA is stable while QDA is more flexible. We build it in 3 steps.

In [ ]:
X_e4 = np.array([[0.0, 0.2], [0.8, 0.1], [1.2, -0.1], [1.8, 0.2], [2.5, 1.5], [3.0, 3.0], [3.5, 2.5], [4.1, 4.0]])  # clouds with different spreads.
y_e4 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # two classes.
_, priors_g_e4, means_g_e4, cov_g_e4 = gda_fit(X_e4, y_e4, shared=True, ridge=1e-3)  # shared covariance.
_, priors_q_e4, means_q_e4, cov_q_e4 = gda_fit(X_e4, y_e4, shared=False, ridge=1e-3)  # separate covariances.
print("shared covariance:\n", np.round(cov_g_e4, 3))  # inspect pooled shape.

▶ What you'll see: GDA averages the spreads from both classes into one covariance.

In [ ]:
print("QDA class covariances:\n", np.round(cov_q_e4, 3))  # inspect separate shapes.
assert cov_q_e4.shape == (2, 2, 2)  # two classes, each with a 2x2 covariance.
print("det shared:", round(float(np.linalg.det(cov_g_e4)), 4))  # compare volume.
print("det class-specific:", np.round([np.linalg.det(c) for c in cov_q_e4], 4))  # class volumes.

In [ ]:
plt.figure(figsize=(4.5, 3.3))  # create determinant comparison.
plt.bar(["GDA shared", "QDA class0", "QDA class1"], [np.linalg.det(cov_g_e4), np.linalg.det(cov_q_e4[0]), np.linalg.det(cov_q_e4[1])], color=["gray", "steelblue", "darkorange"])  # compare covariance volumes.
plt.title("Easy 4: covariance volume comparison"); plt.ylabel("determinant"); plt.xticks(rotation=15); plt.show()  # display.

▶ What you'll see: class-specific determinants can differ substantially, while GDA uses one compromise.

👀 Takeaway: QDA can model unequal class spreads, but GDA pools information for a lower-variance estimate.

### Easy 5 — Use a diagonal ridge for stability

**Goal.** Add $\epsilon I$ before inversion, because covariance matrices from tiny or redundant data can be singular. We build it in 3 steps.

In [ ]:
X_e5 = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])  # perfectly collinear features.
centered_e5 = X_e5 - X_e5.mean(axis=0)  # center before covariance.
S_e5 = centered_e5.T @ centered_e5 / len(X_e5)  # singular covariance estimate.
print("covariance:\n", np.round(S_e5, 3))  # inspect singular matrix.
print("det:", round(float(np.linalg.det(S_e5)), 8))  # determinant shows non-invertibility.

▶ What you'll see: determinant is zero because feature 2 is exactly twice feature 1.

In [ ]:
ridges_e5 = np.array([0.001, 0.01, 0.1, 1.0])  # candidate diagonal ridge strengths.
dets_e5 = np.array([np.linalg.det(S_e5 + r * np.eye(2)) for r in ridges_e5])  # determinant after each ridge.
print("regularized determinants:", np.round(dets_e5, 5))  # inspect stability gain.
assert np.all(dets_e5 > 0)  # every positive ridge makes the matrix invertible.

In [ ]:
plt.figure(figsize=(4.5, 3.2))  # create ridge plot.
plt.plot(ridges_e5, dets_e5, marker="o", color="seagreen")  # determinant versus ridge.
plt.xscale("log"); plt.title("Easy 5: ridge makes Σ invertible"); plt.xlabel("ridge ε"); plt.ylabel("det(Σ+εI)"); plt.show()  # display.

▶ What you'll see: determinant rises from zero once a diagonal ridge is added.

👀 Takeaway: covariance regularization is a practical safeguard, especially when features are correlated or data is scarce.

## 🔴 Advanced

### Advanced 1 — Derive GDA's linear log-odds parameters

**Goal.** Compute the explicit linear log-odds coefficients, because two-class shared-covariance GDA is equivalent to a linear posterior model. We build it in 4 steps.

In [ ]:
X_a1 = np.array([[0.2, 1.0], [0.8, 1.3], [1.0, 0.4], [1.4, 1.1], [3.0, 2.7], [3.5, 3.2], [4.0, 2.9], [3.6, 3.8]])  # training features.
y_a1 = np.array([0, 0, 0, 0, 1, 1, 1, 1])  # labels.
_, priors_a1, means_a1, cov_a1 = gda_fit(X_a1, y_a1, shared=True, ridge=1e-6)  # fit GDA.
inv_a1 = np.linalg.inv(cov_a1)  # precision matrix.
print("means:\n", np.round(means_a1, 3))  # inspect centers.

▶ What you'll see: the two means determine the direction of the linear score.

In [ ]:
w_a1 = inv_a1 @ (means_a1[1] - means_a1[0])  # linear coefficient for log P1/P0.
b_a1 = -0.5 * means_a1[1] @ inv_a1 @ means_a1[1] + 0.5 * means_a1[0] @ inv_a1 @ means_a1[0] + np.log(priors_a1[1] / priors_a1[0])  # intercept.
print("w:", np.round(w_a1, 3))  # inspect linear weights.
print("b:", round(float(b_a1), 3))  # inspect intercept.

In [ ]:
x_a1 = np.array([2.2, 2.0])  # test point.
log_odds_linear_a1 = float(w_a1 @ x_a1 + b_a1)  # direct linear log-odds.
scores_a1 = gda_log_scores(x_a1, priors_a1, means_a1, cov_a1)  # full generative log scores.
log_odds_full_a1 = float(scores_a1[0, 1] - scores_a1[0, 0])  # log score difference.
print("linear log-odds:", round(log_odds_linear_a1, 6))  # inspect linear form.
print("full log-odds:", round(log_odds_full_a1, 6))  # inspect generative form.
assert abs(log_odds_linear_a1 - log_odds_full_a1) < 1e-8  # verify algebraic equivalence.

In [ ]:
xx_a1 = np.linspace(-0.2, 4.6, 100)  # x-values for boundary.
yy_a1 = -(w_a1[0] * xx_a1 + b_a1) / w_a1[1]  # solve w·x+b=0.
plt.figure(figsize=(4.8, 3.5))  # create boundary figure.
plt.scatter(X_a1[:, 0], X_a1[:, 1], c=y_a1, cmap="coolwarm", s=60, edgecolor="k")  # training points.
plt.plot(xx_a1, yy_a1, color="black", lw=2)  # log-odds zero line.
plt.title("Advanced 1: explicit linear GDA boundary"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()  # display.

▶ What you'll see: the algebraic boundary matches the generative posterior decision boundary.

👀 Takeaway: GDA estimates a linear classifier by modeling class densities first, not by directly optimizing a discriminative loss.

### Advanced 2 — Show how priors shift the boundary

**Goal.** Change only $\pi_k$ and redraw the boundary, because imbalanced priors move decisions toward the rarer class's cloud. We build it in 4 steps.

In [ ]:
means_a2 = np.array([[0.0, 0.0], [3.0, 0.0]])  # two class centers on one horizontal axis.
cov_a2 = np.eye(2)  # shared spherical covariance.
prior_sets_a2 = [np.array([0.5, 0.5]), np.array([0.8, 0.2]), np.array([0.2, 0.8])]  # balanced and imbalanced priors.
print("prior sets:", prior_sets_a2)  # inspect the scenarios.

▶ What you'll see: only priors change; means and covariance stay fixed.

In [ ]:
boundaries_a2 = []  # store x-coordinate where log-odds is zero on y=0.
for priors_a2 in prior_sets_a2:  # compute one boundary per prior setting.
    inv_a2 = np.linalg.inv(cov_a2)  # precision matrix.
    w_a2 = inv_a2 @ (means_a2[1] - means_a2[0])  # linear coefficient.
    b_a2 = -0.5 * means_a2[1] @ inv_a2 @ means_a2[1] + 0.5 * means_a2[0] @ inv_a2 @ means_a2[0] + np.log(priors_a2[1] / priors_a2[0])  # intercept with prior ratio.
    boundaries_a2.append(float(-b_a2 / w_a2[0]))  # decision point along x-axis.
print("boundary x-locations:", np.round(boundaries_a2, 3))  # inspect shifts.
assert np.allclose(np.round(boundaries_a2, 3), [1.5, 1.962, 1.038])  # verified prior shifts.

In [ ]:
plt.figure(figsize=(5, 3))  # create prior-shift plot.
for x0_a2, lab_a2 in zip(boundaries_a2, ["π=(.5,.5)", "π=(.8,.2)", "π=(.2,.8)"]):  # draw boundary lines.
    plt.axvline(x0_a2, label=lab_a2)  # vertical boundary for each prior.
plt.scatter(means_a2[:, 0], means_a2[:, 1], c=[0, 1], cmap="coolwarm", s=120, edgecolor="k")  # class centers.
plt.ylim(-1, 1); plt.title("Advanced 2: priors shift the GDA boundary"); plt.xlabel("feature 1"); plt.yticks([]); plt.legend(); plt.show()  # display.

▶ What you'll see: a larger class-0 prior moves the boundary right, requiring more evidence to predict class 1.

In [ ]:
x_probe_a2 = np.array([[1.5, 0.0]])  # midpoint between means.
for priors_a2 in prior_sets_a2:  # posterior at the midpoint under each prior.
    probs_a2 = softmax_from_log(gda_log_scores(x_probe_a2, priors_a2, means_a2, cov_a2))  # posterior probabilities.
    print("priors", priors_a2, "posterior", np.round(probs_a2[0], 3))  # inspect prior effect at same x.

▶ What you'll see: at the exact midpoint, posteriors equal the priors because likelihoods match.

👀 Takeaway: priors encode base rates, so they can shift decisions even when the Gaussian geometry is unchanged.

### Advanced 3 — Diagnose Gaussian-assumption mismatch

**Goal.** Compare a Gaussian class with a ring-shaped class, because GDA can struggle when a class is not well summarized by one ellipse. We build it in 4 steps.

In [ ]:
angles_a3 = np.linspace(0, 2 * np.pi, 16, endpoint=False)  # angles for a ring-shaped class.
ring_a3 = np.c_[2.0 * np.cos(angles_a3), 2.0 * np.sin(angles_a3)]  # non-Gaussian ring.
blob_a3 = 0.35 * np.random.default_rng(3).normal(size=(16, 2))  # compact Gaussian-like blob at origin.
X_a3 = np.vstack([ring_a3, blob_a3])  # combine classes.
y_a3 = np.array([0] * len(ring_a3) + [1] * len(blob_a3))  # ring is class 0, blob is class 1.
print("dataset shape:", X_a3.shape)  # inspect size.

▶ What you'll see: one class is a ring and the other is a center blob.

In [ ]:
_, priors_a3, means_a3, cov_a3 = gda_fit(X_a3, y_a3, shared=True, ridge=1e-3)  # fit a single-ellipse-per-class GDA model.
train_probs_a3 = softmax_from_log(gda_log_scores(X_a3, priors_a3, means_a3, cov_a3))  # training posteriors.
pred_a3 = np.argmax(train_probs_a3, axis=1)  # training predictions.
acc_a3 = np.mean(pred_a3 == y_a3)  # training accuracy.
print("training accuracy:", round(float(acc_a3), 3))  # inspect mismatch effect.

In [ ]:
xx_a3, yy_a3 = np.meshgrid(np.linspace(-2.8, 2.8, 100), np.linspace(-2.8, 2.8, 100))  # grid.
grid_a3 = np.c_[xx_a3.ravel(), yy_a3.ravel()]  # grid rows.
prob1_a3 = softmax_from_log(gda_log_scores(grid_a3, priors_a3, means_a3, cov_a3))[:, 1].reshape(xx_a3.shape)  # class-1 posterior.
print("class means:\n", np.round(means_a3, 3))  # inspect why ring center is misleading.
assert abs(float(means_a3[0, 0])) < 1e-10 and abs(float(means_a3[0, 1])) < 1e-10  # ring mean is near center.

In [ ]:
plt.figure(figsize=(4.6, 3.8))  # create mismatch figure.
plt.contourf(xx_a3, yy_a3, prob1_a3, levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.35)  # posterior background.
plt.contour(xx_a3, yy_a3, prob1_a3, levels=[0.5], colors="black")  # decision boundary.
plt.scatter(X_a3[:, 0], X_a3[:, 1], c=y_a3, cmap="coolwarm", s=50, edgecolor="k")  # training data.
plt.title("Advanced 3: one Gaussian cannot describe a ring"); plt.xlabel("feature 1"); plt.ylabel("feature 2"); plt.show()  # display.

▶ What you'll see: the ring's mean lies near the blob, so the one-Gaussian summary is a poor story for that class.

👀 Takeaway: GDA is strongest when each class is roughly elliptical; multimodal or ring-shaped classes need mixtures, features, or another classifier.

### Advanced 4 — Evaluate GDA with a small holdout split

**Goal.** Train GDA on part of a synthetic dataset and evaluate held-out accuracy, because generative fit still needs future-data validation. We build it in 5 steps.

In [ ]:
rng_a4 = np.random.default_rng(4)  # reproducible synthetic data.
class0_a4 = rng_a4.multivariate_normal([0.0, 0.0], [[0.7, 0.2], [0.2, 0.5]], size=30)  # class-0 Gaussian.
class1_a4 = rng_a4.multivariate_normal([2.0, 1.8], [[0.7, 0.2], [0.2, 0.5]], size=30)  # class-1 Gaussian with same covariance.
X_a4 = np.vstack([class0_a4, class1_a4])  # combine features.
y_a4 = np.array([0] * 30 + [1] * 30)  # combine labels.
print("dataset:", X_a4.shape)  # inspect size.

▶ What you'll see: sixty labeled points from two Gaussian classes.

In [ ]:
idx_a4 = rng_a4.permutation(len(y_a4))  # shuffle indices.
train_idx_a4 = idx_a4[:40]  # first forty for training.
test_idx_a4 = idx_a4[40:]  # remaining twenty for testing.
X_train_a4, y_train_a4 = X_a4[train_idx_a4], y_a4[train_idx_a4]  # training split.
X_test_a4, y_test_a4 = X_a4[test_idx_a4], y_a4[test_idx_a4]  # holdout split.
print("train/test sizes:", len(y_train_a4), len(y_test_a4))  # inspect split sizes.

In [ ]:
_, priors_a4, means_a4, cov_a4 = gda_fit(X_train_a4, y_train_a4, shared=True, ridge=1e-3)  # fit on train only.
train_pred_a4 = np.argmax(softmax_from_log(gda_log_scores(X_train_a4, priors_a4, means_a4, cov_a4)), axis=1)  # train predictions.
test_pred_a4 = np.argmax(softmax_from_log(gda_log_scores(X_test_a4, priors_a4, means_a4, cov_a4)), axis=1)  # test predictions.
train_acc_a4 = np.mean(train_pred_a4 == y_train_a4)  # training accuracy.
test_acc_a4 = np.mean(test_pred_a4 == y_test_a4)  # holdout accuracy.
print("train accuracy:", round(float(train_acc_a4), 3), "test accuracy:", round(float(test_acc_a4), 3))  # inspect generalization.
assert test_acc_a4 >= 0.8  # this seeded Gaussian split is separable enough for GDA.

In [ ]:
cm_a4 = np.zeros((2, 2), dtype=int)  # confusion matrix rows=true, columns=pred.
for t_a4, p_a4 in zip(y_test_a4, test_pred_a4):  # accumulate holdout outcomes.
    cm_a4[t_a4, p_a4] += 1  # increment one cell.
print("confusion matrix:\n", cm_a4)  # inspect mistakes by class.

In [ ]:
plt.figure(figsize=(4, 3))  # create accuracy plot.
plt.bar(["train", "holdout"], [train_acc_a4, test_acc_a4], color=["gray", "seagreen"])  # compare split performance.
plt.ylim(0, 1.05); plt.title("Advanced 4: GDA validation accuracy"); plt.ylabel("accuracy"); plt.show()  # display.

▶ What you'll see: held-out accuracy is close to training accuracy when the Gaussian assumption matches the data.

👀 Takeaway: even a model with closed-form estimates should be checked on data not used to estimate its parameters.

### Advanced 5 — Sweep covariance ridge strength

**Goal.** Tune the diagonal ridge on a high-correlation dataset, because too little ridge is unstable and too much ridge washes out covariance structure. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(5)  # reproducible data.
base0_a5 = rng_a5.normal(0.0, 1.0, size=24)  # latent coordinate for class 0.
base1_a5 = rng_a5.normal(2.0, 1.0, size=24)  # latent coordinate for class 1.
class0_a5 = np.c_[base0_a5, base0_a5 + 0.05 * rng_a5.normal(size=24)]  # highly correlated features.
class1_a5 = np.c_[base1_a5, base1_a5 + 0.05 * rng_a5.normal(size=24)]  # highly correlated features shifted upward.
X_a5 = np.vstack([class0_a5, class1_a5])  # combine features.
y_a5 = np.array([0] * 24 + [1] * 24)  # labels.
print("correlation:", round(float(np.corrcoef(X_a5.T)[0, 1]), 3))  # inspect redundancy.

▶ What you'll see: the two features are almost perfectly correlated.

In [ ]:
idx_a5 = rng_a5.permutation(len(y_a5))  # shuffled split.
train_a5 = idx_a5[:32]  # training indices.
test_a5 = idx_a5[32:]  # holdout indices.
ridges_a5 = np.array([1e-6, 1e-4, 1e-2, 1e-1, 1.0])  # ridge values to compare.
accs_a5 = []  # store holdout accuracies.
conds_a5 = []  # store covariance condition numbers.
print("ridges:", ridges_a5)  # inspect sweep grid.

In [ ]:
for ridge_a5 in ridges_a5:  # train/evaluate one GDA per ridge value.
    _, priors_a5, means_a5, cov_a5 = gda_fit(X_a5[train_a5], y_a5[train_a5], shared=True, ridge=ridge_a5)  # fit ridge-stabilized GDA.
    pred_a5 = np.argmax(softmax_from_log(gda_log_scores(X_a5[test_a5], priors_a5, means_a5, cov_a5)), axis=1)  # holdout predictions.
    accs_a5.append(float(np.mean(pred_a5 == y_a5[test_a5])))  # holdout accuracy.
    conds_a5.append(float(np.linalg.cond(cov_a5)))  # condition number of covariance.
print("accuracies:", np.round(accs_a5, 3))  # inspect validation results.
print("condition numbers:", np.round(conds_a5, 1))  # inspect numerical stability.

In [ ]:
best_i_a5 = int(np.argmax(accs_a5))  # choose highest holdout accuracy.
best_ridge_a5 = float(ridges_a5[best_i_a5])  # selected ridge.
print("best ridge:", best_ridge_a5, "best accuracy:", round(accs_a5[best_i_a5], 3))  # inspect selected setting.
assert max(accs_a5) >= 0.75  # seeded problem has learnable separation.

In [ ]:
fig_a5, ax_a5 = plt.subplots(1, 2, figsize=(7, 3))  # two diagnostics side by side.
ax_a5[0].plot(ridges_a5, accs_a5, marker="o", color="seagreen")  # accuracy curve.
ax_a5[0].set_xscale("log"); ax_a5[0].set_title("holdout accuracy"); ax_a5[0].set_xlabel("ridge ε"); ax_a5[0].set_ylim(0, 1.05)  # label first plot.
ax_a5[1].plot(ridges_a5, conds_a5, marker="o", color="crimson")  # condition curve.
ax_a5[1].set_xscale("log"); ax_a5[1].set_yscale("log"); ax_a5[1].set_title("condition number"); ax_a5[1].set_xlabel("ridge ε")  # label second plot.
plt.suptitle("Advanced 5: ridge trades stability and fit"); plt.tight_layout(); plt.show()  # display.

▶ What you'll see: larger ridge values improve numerical conditioning, while accuracy may peak before the strongest ridge.

👀 Takeaway: ridge strength is a stability knob; tune it with validation data rather than assuming zero or a huge value is best.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

GDA learns class priors and Gaussian feature distributions, then applies Bayes' rule.

Gaussian Discriminant Analysis builds predictions from likelihoods and priors through Bayes' rule. The notebook keeps the lesson grounded by checking the exact average loss, cost, and validation gap before scaling to real data.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def lesson_score(losses, cost, alternative):
    losses = np.asarray(losses, dtype=float)
    raw = round(float(losses.mean()), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    return {
        "losses": losses,
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
    }


def binary_logistic_train(x_tr, y_tr, lr=0.2, steps=900, l2=0.02):
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    weights = np.zeros(X.shape[1])
    y = y_tr.astype(float)
    for step in range(steps):
        logits = X @ weights
        probs = 1.0 / (1.0 + np.exp(-logits))
        grad = X.T @ (probs - y) / y.size
        grad[1:] = grad[1:] + l2 * weights[1:]
        weights = weights - lr * grad
    return weights


def binary_logistic_predict(weights, x_te):
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    probs = 1.0 / (1.0 + np.exp(-(X @ weights)))
    return (probs >= 0.5).astype(int)


def softmax_train(x_tr, y_tr, lr=0.15, steps=1100, l2=0.01):
    classes = np.unique(y_tr)
    mapping = {label: idx for idx, label in enumerate(classes)}
    y_idx = np.array([mapping[label] for label in y_tr])
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    W = np.zeros((X.shape[1], classes.size))
    Y = np.eye(classes.size)[y_idx]
    for step in range(steps):
        logits = X @ W
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_scores = np.exp(logits)
        probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        grad = X.T @ (probs - Y) / X.shape[0]
        grad[1:, :] = grad[1:, :] + l2 * W[1:, :]
        W = W - lr * grad
    return W, classes


def softmax_predict(model, x_te):
    W, classes = model
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    logits = X @ W
    return classes[np.argmax(logits, axis=1)]


def glm_predict(x_tr, y_tr, x_te):
    classes = np.unique(y_tr)
    if classes.size == 2:
        weights = binary_logistic_train(x_tr, y_tr)
        return binary_logistic_predict(weights, x_te)
    model = softmax_train(x_tr, y_tr)
    return softmax_predict(model, x_te)


def lda_qda_predict(x_tr, y_tr, x_te, mode="lda"):
    classes, counts = np.unique(y_tr, return_counts=True)
    if x_tr.shape[0] <= classes.size or counts.min() < 2:
        model = GaussianNB(var_smoothing=1e-8)
    elif mode == "qda":
        model = QuadraticDiscriminantAnalysis(reg_param=0.08)
    else:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def gda_predict(x_tr, y_tr, x_te):
    model = GaussianNB(var_smoothing=1e-8)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def sklearn_logistic_predict(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2500)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def classifier_metrics(rungs, predictor):
    rows = []
    for level, item in enumerate(rungs, start=1):
        name, X, y = item
        accuracy = clf_accuracy(predictor, X, y)
        rows.append({"level": level, "name": name, "accuracy": float(accuracy)})
    return rows


def plot_classifier_summary(rungs, rows, predictor):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, item, row in zip(axes[:5], rungs, rows):
        name, X, y = item
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        preds = predictor(x_tr_s, y_tr, x_te_s)
        ax.scatter(x_te_s[:, 0], x_te_s[:, 1], c=preds, s=14, cmap="viridis", alpha=0.8)
        ax.set_title(f"D{row['level']} acc={row['accuracy']:.2f}")
        ax.set_xlabel("feature 0")
        ax.set_ylabel("feature 1")
    axes[5].plot([row["level"] for row in rows], [row["accuracy"] for row in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_title("Accuracy vs ladder rung")
    axes[5].set_xlabel("D1 to D5")
    axes[5].set_ylabel("held-out accuracy")
    plt.tight_layout()
    plt.show()

def logistic_regression_method(losses=None, cost=0.070, alternative=0.394):
    if losses is None:
        losses = np.array([0.235, 0.109, 0.471])
    return lesson_score(losses, cost, alternative)


def softmax_multinomial_regression_method(losses=None, cost=0.080, alternative=0.401):
    if losses is None:
        losses = np.array([0.246, 0.122, 0.488])
    return lesson_score(losses, cost, alternative)


def generalized_linear_models_method(losses=None, cost=0.090, alternative=0.429):
    if losses is None:
        losses = np.array([0.257, 0.135, 0.505])
    return lesson_score(losses, cost, alternative)


def linear_quadratic_discriminant_analysis_method(losses=None, cost=0.100, alternative=0.457):
    if losses is None:
        losses = np.array([0.268, 0.148, 0.522])
    return lesson_score(losses, cost, alternative)


def gaussian_discriminant_analysis_method(losses=None, cost=0.050, alternative=0.361):
    if losses is None:
        losses = np.array([0.180, 0.070, 0.539])
    return lesson_score(losses, cost, alternative)

## The concept, built once (D1)

The lesson formula is

$$p(y=k\mid x)=\frac{p(x\mid y=k)\pi_k}{\sum_j p(x\mid y=j)\pi_j}$$

For D1, the verified per-example losses are 0.180, 0.070, 0.539. The empirical risk is the average, and the model-selection score is that raw term plus the lesson cost.

In [ ]:
result = gaussian_discriminant_analysis_method()
print(result)
assert np.isclose(result["raw"], 0.263)
assert np.isclose(result["score"], 0.313)
assert np.isclose(result["gap"], 0.048)

The exact arithmetic is $R_S=(0.180, 0.070, 0.539)/3=0.263$, then $score=R_S+0.050=0.313$. The tempting alternative is 0.361, so the validation gap is $0.361-0.313=0.048$.

In [ ]:
stable_score = 0.80 * result["score"]
relative_gap = result["gap"] / result["alternative"]
print(f"stable={stable_score:.3f} relative_gap={relative_gap:.3f}")
assert stable_score < result["score"]
assert relative_gap > 0.0

## The dataset ladder

The same method now runs on D1 through D5. The printed preview shows shape, class balance or target scale, and a small sample before any fitting.

In [ ]:
rungs = clf_ladder()
for level, item in enumerate(rungs, start=1):
    name, X, y = item
    labels, counts = np.unique(y, return_counts=True)
    class_info = dict(zip(labels.tolist(), counts.tolist()))
    print(f"D{level}: {name} X={X.shape} classes={class_info}")
    print("sample X", np.round(X[:3, : min(3, X.shape[1])], 3))
    print("sample y", y[:3])

## Run the same method across D1-D5

The single headline metric for this lesson is accuracy. Classification topics also print the shared logistic baseline for a no-special-skill comparison.

In [ ]:
predictor = gda_predict
rows = classifier_metrics(rungs, predictor)
print("rung | accuracy | logistic baseline | dataset")
for row, item in zip(rows, rungs):
    name, X, y = item
    baseline = clf_accuracy(logistic_baseline, X, y)
    print(f"D{row['level']} | {row['accuracy']:.3f} | {baseline:.3f} | {row['name']}")
assert len(rows) == 5
assert all(0.0 <= row["accuracy"] <= 1.0 for row in rows)

## Results visualization

The closing figure has two parts: small multiples for each rung and one summary curve from D1 to D5.

In [ ]:
plot_classifier_summary(rungs, rows, predictor)

## Pitfall on the hardest rung

Pitfall: optimizing the raw term and forgetting the cost. The wrong check looks only at the raw D5 metric; the fix restores the lesson's cost and gap before selecting a winner.

In [ ]:
name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
main_preds = predictor(x_tr, y_tr, x_te)
base_preds = logistic_baseline(x_tr, y_tr, x_te)
main_acc = accuracy_score(y_te, main_preds)
base_acc = accuracy_score(y_te, base_preds)
lesson = gaussian_discriminant_analysis_method()
raw_only = 1.0 - main_acc
full_score = raw_only + lesson["cost"]
alt_score = (1.0 - base_acc) + lesson["alternative"]
print(f"Raw-only D5 error={raw_only:.3f} baseline_error={1.0 - base_acc:.3f}")
print(f"Full score with lesson cost={full_score:.3f} alternative score={alt_score:.3f}")
print(f"Lesson cost={lesson['cost']:.3f} gap={lesson['gap']:.3f}")
print(f"Macro-F1 sanity={f1_score(y_te, main_preds, average='macro'):.3f}")
assert lesson["gap"] > 0.0
assert 0.0 <= full_score

## Evaluate it + Practice

- Compare the headline metric with the no-skill or logistic baseline before claiming improvement.
- Run a cheap sanity check: shuffled labels should damage accuracy, and injected outliers should make robust regression matter.
- Ablation: turn off the key idea, such as Huber clipping, softmax normalization, the GLM link, covariance modeling, or Bayes priors; the D5 metric should usually drop.
- Failure signals include unstable validation gaps, wildly different scales, singular covariance warnings, or a D5 score that only wins before cost is included.

Practice 1: change the cost term and recompute the decision score.

Practice 2: rerun D5 after removing one informative feature group and compare the metric.

Practice 3: create a shuffled-label baseline and explain why it should fail.